# KBB notlarini PDF olarak Drive'a yaz

Notlarin HTML kaynagi, baski stili ve gorsel listesi git deposunda durur.
Bu defter her seyi kendisi halleder:

1. Depoyu ceker
2. Eski PDF'lerden korunmasi gereken fotograflari cikarir
3. Internetten gelen gorselleri indirip Drive'a **dosya olarak** kaydeder
4. Butun notlari PDF'e cevirip yerine yazar:
   - gun notlari -> `KBB_not_claude/`
   - `soru-` ile baslayanlar -> `KBB_not_claude/Soru/`

**Yapman gereken tek sey: Calisma Zamani -> Tumunu calistir (Ctrl+F9).**
Elle duzenlenecek hicbir sey yok. Degismemis notlar atlanir, indirilmis
gorseller tekrar indirilmez - yani ikinci calistirma cok hizlidir.

Yeni not eklendiginde de ayni: defteri bir kez calistir, hepsi yerine oturur.

In [ ]:
# 1) Drive'i bagla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Araclari kur
!pip install -q weasyprint pymupdf requests
!apt-get -qq install -y fonts-liberation > /dev/null
print("kurulum tamam")

In [ ]:
# 3) Depoyu cek, klasorleri bul
import glob, os, shutil, subprocess, json

DEPO  = "https://github.com/alpercil/Repository1"
YEREL = "/content/repo"

if os.path.exists(YEREL):
    shutil.rmtree(YEREL)
subprocess.run(["git", "clone", "--depth", "1", "-q", DEPO, YEREL], check=True)

HEDEF = "/content/drive/MyDrive/PAÜ/KBB_not_claude"
if not os.path.isdir(HEDEF):
    adaylar = glob.glob("/content/drive/MyDrive/**/KBB_not_claude", recursive=True)
    if not adaylar:
        raise SystemExit("KBB_not_claude bulunamadi - HEDEF'i elle yaz")
    HEDEF = adaylar[0]

SORU   = os.path.join(HEDEF, "Soru");   os.makedirs(SORU, exist_ok=True)
GORSEL = os.path.join(HEDEF, "gorsel"); os.makedirs(GORSEL, exist_ok=True)
KAYNAK = os.path.join(YEREL, "kbb", "notlar")

# Gorseller Drive'da durur; render sirasinda HTML'in yaninda gorunmeleri icin
# calisma kopyasina baglanti kurulur.
BAG = os.path.join(KAYNAK, "gorsel")
if os.path.islink(BAG) or os.path.exists(BAG):
    os.remove(BAG) if os.path.islink(BAG) else shutil.rmtree(BAG)
os.symlink(GORSEL, BAG)

NOTLAR = sorted(glob.glob(os.path.join(KAYNAK, "*.html")))
print("hedef  :", HEDEF)
print("gorsel :", GORSEL)
print("not    :", len(NOTLAR), "HTML")

In [ ]:
# 4) Eski PDF'lerden korunacak fotograflari cikar
#
# HTML'de "gorsel/<not>/rNN.png" diye bir referans varsa ve dosya yoksa,
# ayni adli eski PDF'ten cikarilir. Zaten varsa dokunulmaz.

import re
import pymupdf

istenen = {}   # not adi -> en buyuk rNN numarasi
for html in NOTLAR:
    ad = os.path.splitext(os.path.basename(html))[0]
    metin = open(html, encoding="utf-8").read()
    for m in re.finditer(rf"gorsel/{re.escape(ad)}/r(\d+)\.png", metin):
        istenen[ad] = max(istenen.get(ad, 0), int(m.group(1)))

for ad, gereken in istenen.items():
    klasor = os.path.join(GORSEL, ad)
    os.makedirs(klasor, exist_ok=True)
    varolan = len(glob.glob(os.path.join(klasor, "r*.png")))
    if varolan >= gereken:
        print(f"  atlandi (hazir): {ad} - {varolan} fotograf")
        continue

    eski = os.path.join(HEDEF, ad + ".pdf")
    if not os.path.exists(eski):
        print(f"  UYARI: {ad}.pdf yok, fotograflar cikarilamadi")
        continue

    belge = pymupdf.open(eski)
    sayac, gorulen = 0, set()
    for sayfa in belge:
        for bilgi in sayfa.get_images(full=True):
            xref = bilgi[0]
            if xref in gorulen:
                continue
            gorulen.add(xref)
            pix = pymupdf.Pixmap(belge, xref)
            if pix.n - pix.alpha >= 4:              # CMYK -> RGB
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix)
            if pix.width < 120 or pix.height < 120:  # cizgi/ikon artiklarini ele
                continue
            sayac += 1
            pix.save(os.path.join(klasor, f"r{sayac:02d}.png"))
    print(f"  {ad}: {sayac} fotograf cikarildi")

if not istenen:
    print("  eski PDF'ten kurtarilacak fotograf yok")

In [ ]:
# 5) Internetten gelen gorselleri indir ve Drive'a dosya olarak yaz
#
# Hangi notta hangi gorselin hangi yerel ada inecegi kbb/notlar/gorseller.json
# dosyasinda yazili. Inmis olanlar tekrar indirilmez.

import requests

MANIFEST = os.path.join(KAYNAK, "gorseller.json")
kayit = json.load(open(MANIFEST, encoding="utf-8")) if os.path.exists(MANIFEST) else {}

BASLIK = {"User-Agent": "KBB-calisma-notu/1.0 (kisisel egitim amacli)"}
sorunlu = []

for ad, gorseller in kayit.items():
    klasor = os.path.join(GORSEL, ad)
    os.makedirs(klasor, exist_ok=True)
    for g in gorseller:
        hedef = os.path.join(klasor, g["ad"])
        if os.path.exists(hedef) and os.path.getsize(hedef) > 1024:
            print(f"  atlandi (var): {ad}/{g['ad']}")
            continue
        try:
            y = requests.get(g["url"], headers=BASLIK, timeout=60, allow_redirects=True)
            y.raise_for_status()
            if not y.headers.get("content-type", "").startswith("image/"):
                raise ValueError("gorsel degil: " + y.headers.get("content-type", "?"))
            with open(hedef, "wb") as f:
                f.write(y.content)
            print(f"  indi: {ad}/{g['ad']}  {len(y.content)/1024:.0f} KB  <- {g['commons']}")
        except Exception as e:
            sorunlu.append((ad, g["ad"], g["commons"], str(e)))
            print(f"  HATA: {ad}/{g['ad']}  <- {g['commons']}  ({e})")

if sorunlu:
    print("\nInmeyen gorseller - bunlari Claude'a bildir, degistirsin:")
    for n, a, c, e in sorunlu:
        print(f"  {n} / {a} / {c}")
else:
    print("\nButun gorseller yerinde.")

In [ ]:
# 6) Notlari PDF'e cevir
from weasyprint import HTML

for html in NOTLAR:
    ad = os.path.splitext(os.path.basename(html))[0]
    klasor = SORU if ad.startswith("soru-") else HEDEF
    cikti = os.path.join(klasor, ad + ".pdf")

    # HTML degismisse yenile; degismemisse atla
    if os.path.exists(cikti) and os.path.getmtime(cikti) >= os.path.getmtime(html):
        print(f"  atlandi (guncel): {ad}.pdf")
        continue

    HTML(html, base_url=KAYNAK).write_pdf(cikti)
    nere = "Soru/" if klasor is SORU else ""
    print(f"  uretildi: {nere}{ad}.pdf  {os.path.getsize(cikti)/1e6:.2f} MB")

print("\nBitti.")